# Exploración de APIs

**football-data.org** → partidos, resultados, clasificación  
**Open-Meteo** → clima histórico por coordenadas

In [1]:
import requests
import json
import time
import pandas as pd
import os

from pprint import pprint
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

FOOTBALL_API_KEY = os.getenv("FOOTBALL_API_KEY")
if not FOOTBALL_API_KEY:
    raise ValueError("FOOTBALL_API_KEY no está definida en el .env")

HEADERS = {"X-Auth-Token": FOOTBALL_API_KEY}

---
## 1. football-data.org
### 1.1 ¿Qué competiciones tenemos disponibles?

In [2]:
r = requests.get("https://api.football-data.org/v4/competitions", headers=HEADERS)
print("Status:", r.status_code)
data = r.json()

# Ver las competiciones
for comp in data["competitions"]:
    print(f"{comp['code']:10} | {comp['name']}")

Status: 200
BSA        | Campeonato Brasileiro Série A
ELC        | Championship
PL         | Premier League
CL         | UEFA Champions League
EC         | European Championship
FL1        | Ligue 1
BL1        | Bundesliga
SA         | Serie A
DED        | Eredivisie
PPL        | Primeira Liga
CLI        | Copa Libertadores
PD         | Primera Division
WC         | FIFA World Cup


### 1.2 Partidos de La Liga (PD) temporada 2023
Primero miramos cuántos partidos hay y cómo está estructurado cada partido.

In [3]:
time.sleep(6)  # Limite de 10 requests por minuto

r = requests.get(
    "https://api.football-data.org/v4/competitions/PD/matches",
    headers=HEADERS,
    params={"season": 2023}
)
print("Status:", r.status_code)
matches_raw = r.json()

print(f"Total partidos: {matches_raw['resultSet']['count']}")
print(f"Temporada: {matches_raw['resultSet']['first']} → {matches_raw['resultSet']['last']}")

Status: 200
Total partidos: 380
Temporada: 2023-08-11 → 2024-05-26


In [4]:
# Miramos cómo es un partido por dentro
primer_partido = matches_raw["matches"][0]
pprint(primer_partido)

{'area': {'code': 'ESP',
          'flag': 'https://crests.football-data.org/760.svg',
          'id': 2224,
          'name': 'Spain'},
 'awayTeam': {'crest': 'https://crests.football-data.org/87.png',
              'id': 87,
              'name': 'Rayo Vallecano de Madrid',
              'shortName': 'Rayo Vallecano',
              'tla': 'RAY'},
 'competition': {'code': 'PD',
                 'emblem': 'https://crests.football-data.org/laliga.png',
                 'id': 2014,
                 'name': 'Primera Division',
                 'type': 'LEAGUE'},
 'group': None,
 'homeTeam': {'crest': 'https://crests.football-data.org/267.png',
              'id': 267,
              'name': 'UD Almería',
              'shortName': 'Almería',
              'tla': 'ALM'},
 'id': 438482,
 'lastUpdated': '2023-10-09T15:20:25Z',
 'matchday': 1,
 'odds': {'msg': 'Activate Odds-Package in User-Panel to retrieve odds.'},
 'referees': [{'id': 80747,
               'name': 'Javier Alberola Rojas',
 

In [5]:
# Campos que se quieren obtener de cada partido
print("ID:",          primer_partido["id"])
print("Fecha:",       primer_partido["utcDate"])
print("Estado:",      primer_partido["status"])         # FINISHED, SCHEDULED, etc.
print("Local:",       primer_partido["homeTeam"]["name"])
print("Visitante:",   primer_partido["awayTeam"]["name"])
print("Resultado:",   primer_partido["score"]["fullTime"])
print("Ganador:",     primer_partido["score"]["winner"]) # HOME_TEAM / AWAY_TEAM / DRAW / None

ID: 438482
Fecha: 2023-08-11T17:30:00Z
Estado: FINISHED
Local: UD Almería
Visitante: Rayo Vallecano de Madrid
Resultado: {'home': 0, 'away': 2}
Ganador: AWAY_TEAM


In [6]:
# Convertimos todos los partidos a DataFrame
rows = []
for m in matches_raw["matches"]:
    rows.append({
        "match_id":   m["id"],
        "date":       m["utcDate"],
        "status":     m["status"],
        "home_team":  m["homeTeam"]["name"],
        "away_team":  m["awayTeam"]["name"],
        "home_score": m["score"]["fullTime"].get("home"),
        "away_score": m["score"]["fullTime"].get("away"),
        "winner":     m["score"].get("winner"),
    })

df_matches = pd.DataFrame(rows)
df_matches["date"] = pd.to_datetime(df_matches["date"])
df_matches.head(10)

,match_id,date,status,home_team,away_team,home_score,away_score,winner
0,438482,2023-08-11 17:30:00+00:00,FINISHED,UD Almería,Rayo Vallecano de Madrid,0,2,AWAY_TEAM
1,438479,2023-08-11 20:00:00+00:00,FINISHED,Sevilla FC,Valencia CF,1,2,AWAY_TEAM
2,438481,2023-08-12 15:00:00+00:00,FINISHED,Real Sociedad de Fútbol,Girona FC,1,1,DRAW
3,438483,2023-08-12 17:30:00+00:00,FINISHED,UD Las Palmas,RCD Mallorca,1,1,DRAW
4,438474,2023-08-12 19:30:00+00:00,FINISHED,Athletic Club,Real Madrid CF,0,2,AWAY_TEAM
5,438476,2023-08-13 15:00:00+00:00,FINISHED,RC Celta de Vigo,CA Osasuna,0,2,AWAY_TEAM
6,438480,2023-08-13 17:30:00+00:00,FINISHED,Villarreal CF,Real Betis Balompié,1,2,AWAY_TEAM
7,438478,2023-08-13 19:30:00+00:00,FINISHED,Getafe CF,FC Barcelona,0,0,DRAW
8,438477,2023-08-14 17:30:00+00:00,FINISHED,Cádiz CF,Deportivo Alavés,1,0,HOME_TEAM
9,438475,2023-08-14 19:30:00+00:00,FINISHED,Club Atlético de Madrid,Granada CF,3,1,HOME_TEAM


In [7]:
# ¿Cuántos partidos hay por estado?
print(df_matches["status"].value_counts())

# ¿Cuántos partidos tienen resultado?
terminados = df_matches[df_matches["status"] == "FINISHED"]
print(f"\nPartidos terminados: {len(terminados)} de {len(df_matches)}")

status
FINISHED    380
Name: count, dtype: int64

Partidos terminados: 380 de 380


In [8]:
# ¿Qué equipos hay y cuántos partidos tienen como local?
print(df_matches["home_team"].value_counts())

home_team
UD Almería                  19
Sevilla FC                  19
Real Sociedad de Fútbol     19
UD Las Palmas               19
Athletic Club               19
RC Celta de Vigo            19
Villarreal CF               19
Getafe CF                   19
Cádiz CF                    19
Club Atlético de Madrid     19
RCD Mallorca                19
Valencia CF                 19
CA Osasuna                  19
Girona FC                   19
FC Barcelona                19
Real Betis Balompié         19
Deportivo Alavés            19
Granada CF                  19
Rayo Vallecano de Madrid    19
Real Madrid CF              19
Name: count, dtype: int64


### 1.3 Clasificación (standings)

In [9]:
time.sleep(6)  # Límite 10 calls/min

r = requests.get(
    "https://api.football-data.org/v4/competitions/PD/standings",
    headers=HEADERS,
    params={"season": 2023}
)
standings_raw = r.json()

# La clasificación viene en 3 tipos: TOTAL, HOME, AWAY
tabla_total = standings_raw["standings"][0]["table"]

df_standings = pd.DataFrame([{
    "position": t["position"],
    "team":     t["team"]["name"],
    "points":   t["points"],
    "won":      t["won"],
    "draw":     t["draw"],
    "lost":     t["lost"],
    "gf":       t["goalsFor"],
    "ga":       t["goalsAgainst"],
} for t in tabla_total])

df_standings

,position,team,points,won,draw,lost,gf,ga
0,1,Real Madrid CF,95,29,8,1,87,26
1,2,FC Barcelona,85,26,7,5,79,44
2,3,Girona FC,81,25,6,7,85,46
3,4,Club Atlético de Madrid,76,24,4,10,70,43
4,5,Athletic Club,68,19,11,8,61,37
5,6,Real Sociedad de Fútbol,60,16,12,10,51,39
6,7,Real Betis Balompié,57,14,15,9,48,45
7,8,Villarreal CF,53,14,11,13,65,65
8,9,Valencia CF,49,13,10,15,40,45
9,10,Deportivo Alavés,46,12,10,16,36,46


---
## 2. Open-Meteo (clima histórico)

In [10]:
# Coordenadas de Madrid (Bernabéu)
params = {
    "latitude":   40.4530,
    "longitude":  -3.6883,
    "start_date": "2023-08-01",
    "end_date":   "2023-09-30",
    "daily": [
        "temperature_2m_max",
        "temperature_2m_min",
        "precipitation_sum",
        "windspeed_10m_max",
        "weathercode"
    ],
    "timezone": "Europe/Madrid"
}

r = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params)
print("Status:", r.status_code)
weather_raw = r.json()
pprint(weather_raw)

Status: 200
{'daily': {'precipitation_sum': [0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,
                                 0.0,


In [11]:
# Convertimos a DataFrame
daily = weather_raw["daily"]
df_weather = pd.DataFrame({
    "date":          pd.to_datetime(daily["time"]),
    "temp_max":      daily["temperature_2m_max"],
    "temp_min":      daily["temperature_2m_min"],
    "temp_avg":      [(mx + mn) / 2 for mx, mn 
                      in zip(daily["temperature_2m_max"], daily["temperature_2m_min"])],
    "precipitation": daily["precipitation_sum"],
    "wind_max":      daily["windspeed_10m_max"],
    "weather_code":  daily["weathercode"],
})
df_weather.head(10)

,date,temp_max,temp_min,temp_avg,precipitation,wind_max,weather_code
0,2023-08-01,36.7,20.1,28.40,0.0,21.7,0
1,2023-08-02,35.4,19.5,27.45,0.0,20.1,1
2,2023-08-03,33.1,20.1,26.60,0.0,21.2,0
3,2023-08-04,28.1,14.9,21.50,0.0,25.0,2
4,2023-08-05,33.7,14.5,24.10,0.0,16.8,3
5,2023-08-06,36.3,18.3,27.30,0.0,21.2,0
6,2023-08-07,36.5,17.4,26.95,0.0,23.0,0
7,2023-08-08,38.6,21.1,29.85,0.0,14.5,1
8,2023-08-09,39.5,22.5,31.00,0.0,23.3,2
9,2023-08-10,38.5,23.8,31.15,0.0,18.1,0


In [ ]:
# Estadísticas básicas del clima
df_weather.describe()

---
## 3. Join de prueba: partidos + clima
Comprobamos que el join funciona antes de escribirlo en el transformer.

In [12]:
# Filtramos partidos de Madrid en el rango de fechas que tenemos de clima
df_matches["date"] = pd.to_datetime(df_matches["date"], utc=True).dt.tz_localize(None)
df_matches["date_only"] = df_matches["date"].dt.date
df_weather["date_only"] = df_weather["date"].dt.date

# Partidos del Real Madrid como local (ciudad: Madrid)
madrid_home = df_matches[
    df_matches["home_team"].str.contains("Real Madrid", case=False)
].copy()

merged = madrid_home.merge(df_weather, on="date_only", how="left")
merged[["date_only", "home_team", "away_team", "winner", "temp_avg", "precipitation"]].head(10)

,date_only,home_team,away_team,winner,temp_avg,precipitation
0,2023-09-02,Real Madrid CF,Getafe CF,HOME_TEAM,19.15,6.6
1,2023-09-17,Real Madrid CF,Real Sociedad de Fútbol,HOME_TEAM,17.40,18.3
2,2023-09-27,Real Madrid CF,UD Las Palmas,HOME_TEAM,20.25,0.0
3,2023-10-07,Real Madrid CF,CA Osasuna,HOME_TEAM,NaN,NaN
4,2023-11-05,Real Madrid CF,Rayo Vallecano de Madrid,DRAW,NaN,NaN
5,2023-11-11,Real Madrid CF,Valencia CF,HOME_TEAM,NaN,NaN
6,2023-12-02,Real Madrid CF,Granada CF,HOME_TEAM,NaN,NaN
7,2023-12-17,Real Madrid CF,Villarreal CF,HOME_TEAM,NaN,NaN
8,2024-01-03,Real Madrid CF,RCD Mallorca,HOME_TEAM,NaN,NaN
9,2024-01-21,Real Madrid CF,UD Almería,HOME_TEAM,NaN,NaN


In [ ]:
# ¿Cuántos partidos quedaron sin clima? (fechas fuera del rango descargado)
print(f"Partidos sin clima: {merged['temp_avg'].isna().sum()} de {len(merged)}")
# Si hay muchos NaN, ampliar el rango de fechas en Open-Meteo

---
## 4. Conclusiones para el extractor

Después de explorar, anota aquí lo que has aprendido:

- Campos útiles de football-data: `id`, `utcDate`, `status`, `homeTeam.name`, `awayTeam.name`, `score.fullTime`, `score.winner`
- Partidos terminados: usar `status == 'FINISHED'`
- El `winner` puede ser `None` si el partido no ha terminado — manejar esto
- Open-Meteo devuelve datos diarios, el join es por fecha
- Hay que descargar el clima para el rango completo de la temporada (agosto 2023 → junio 2024)
- **Límite API football**: `time.sleep(6)` entre llamadas para no superar 10/min